# NB0 — SFT Dataset Builder

**No GPU required.** Run this before `sft_combined_all6.ipynb`.

## What this builds
This notebook constructs three JSONL files saved as Kaggle Dataset `sft-datasets`:

| File | Contents | Used by |
|------|---------|---------|
| `control_sft.jsonl` | ~16,600 Alpaca examples (low quality baseline) | E1, E3 |
| `golden_sft.jsonl` | ~24,000 curated examples from LIMA + OpenHermes + Dolly with LFR difficulty scores | E2, E4, E5, E6 |
| `val_sft.jsonl` | 300 LIMA test examples (neutral validation — never in training) | all experiments |

## Design rationale
- **Control**: Stanford Alpaca 52K (GPT-3 generated, inconsistent quality) — the realistic baseline
- **Golden**: LIMA (hand-curated) + OpenHermes-2.5 filtered (GPT-4) + Dolly-15K filtered (human written)
- **Quality filter**: response length ≥ 80 words + reasoning marker density ≥ 2 + no formatting artifacts
- **Difficulty scoring**: 3-component lexical score → percentile split into easy/medium/hard (LFR phases)
- **Val set**: LIMA test split — distribution-neutral, never contaminated by training

## After running
Save output (`/kaggle/working/`) as Kaggle Dataset named exactly `sft-datasets`.
The training notebook mounts this at `/kaggle/input/sft-datasets/`.

In [1]:
# CELL 1 — Install dependencies (no GPU needed)
import subprocess, sys

PKGS = [
    "datasets>=2.14.0",
    "huggingface_hub>=0.19.0",
    "tqdm",
    "numpy",
]
for p in PKGS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=True)
    print(f"  ✅ {p}")

print("\nInstall complete.")

  ✅ datasets>=2.14.0
  ✅ huggingface_hub>=0.19.0
  ✅ tqdm
  ✅ numpy

Install complete.


In [2]:
# CELL 2 — HF login + output dir
import os

WRITE_DIR = "/kaggle/working"
os.makedirs(WRITE_DIR, exist_ok=True)

# HF login — needed for gated datasets (LIMA, LLaMA tokeniser if used later)
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("HF login : ✅")
except Exception as e:
    print(f"HF login : ⚠️  ({e}) — public datasets will still work")

print(f"Output dir: {WRITE_DIR}")

HF login : ⚠️  (Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 114594827 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.) — public datasets will still work
Output dir: /kaggle/working


In [3]:
# CELL 3 — Configuration
import random
random.seed(42)

# ── Token budget targets (must match sft_combined_all6.ipynb) ──
# TOKEN_BUDGET in training notebook = 2_000_000
# At ~180 tokens/example avg, 2M tokens ≈ 11,000 examples
# We build a larger pool so the training notebook can sample what it needs

# Control: how many Alpaca examples to keep (pool — training samples subset)
CONTROL_TARGET = 20_000   # ~2.4M tokens at avg 120 tok/example

# Golden: how many curated examples to keep
GOLDEN_TARGET  = 24_000   # mix of LIMA + OpenHermes + Dolly

# OpenHermes sample size (from 1M total — we filter then sample)
OH_SAMPLE      = 25_000   # sample before quality filtering
OH_KEEP_MAX    = 20_000   # max after filtering

# Dolly filter: keep only these substantive categories
DOLLY_CATEGORIES = ["closed_qa", "open_qa", "information_extraction", "summarization"]
DOLLY_MIN_WORDS  = 100

# Quality filter thresholds (golden only)
MIN_RESPONSE_WORDS   = 80    # substantive answers only
MIN_REASONING_MARKERS = 2    # must show reasoning structure

# Difficulty scoring weights (LFR phases)
WEIGHT_COMPLEXITY  = 0.35  # response length + reasoning density
WEIGHT_AMBIGUITY   = 0.25  # instruction under-specification
WEIGHT_PPL_PROXY   = 0.40  # vocabulary richness (type-token ratio)

# Validation set size
VAL_SIZE = 300

print("Config:")
print(f"  Control target : {CONTROL_TARGET:,} examples")
print(f"  Golden target  : {GOLDEN_TARGET:,} examples")
print(f"  Val set size   : {VAL_SIZE}")

Config:
  Control target : 20,000 examples
  Golden target  : 24,000 examples
  Val set size   : 300


In [4]:
# CELL 4 — Quality filter + difficulty scorer (no model — fully lexical)
import re
from collections import Counter

REASONING_MARKERS = [
    "because", "therefore", "however", "consequently",
    "which means", "as a result", "in contrast",
    "on the other hand", "first", "second", "third",
    "finally", "in summary", "in conclusion", "for example",
    "specifically", "moreover", "furthermore", "thus",
]

ARTIFACTS = [
    "[/INST]", "<<SYS>>", "USER:", "ASSISTANT:",
    "<s>", "</s>", "[INST]", "Human:", "AI:",
    "http://", "https://", "www.",  # bare URLs are low quality
]


def passes_quality_filter(response: str) -> bool:
    """Return True if response meets golden quality bar."""
    words = response.split()

    # Minimum length
    if len(words) < MIN_RESPONSE_WORDS:
        return False

    # Reasoning density
    text_lower = response.lower()
    marker_hits = sum(text_lower.count(m) for m in REASONING_MARKERS)
    if marker_hits < MIN_REASONING_MARKERS:
        return False

    # No formatting artifacts
    if any(a in response for a in ARTIFACTS):
        return False

    return True


def difficulty_score(instruction: str, response: str) -> dict:
    """
    Three-component lexical difficulty score [0, 1].
    No base model needed — fully deterministic.

    Component 1 — Response complexity (length + reasoning density)
    Component 2 — Instruction ambiguity (under-specification)
    Component 3 — Vocabulary richness as PPL proxy (type-token ratio)
    """
    resp_words = response.split()
    resp_sents = [s for s in response.split(".") if s.strip()]

    # 1. Complexity
    text_lower = response.lower()
    marker_density = sum(text_lower.count(m) for m in REASONING_MARKERS) / max(len(resp_sents), 1)
    length_score   = min(len(resp_words) / 500, 1.0)   # cap at 500 words
    complexity     = 0.5 * length_score + 0.5 * min(marker_density / 3.0, 1.0)

    # 2. Ambiguity (inverse of instruction specificity)
    instr_words   = len(instruction.split())
    specificity   = min(instr_words / 30, 1.0)
    ambiguity     = 1.0 - specificity

    # 3. Vocabulary richness (type-token ratio) as difficulty proxy
    words_lower = [w.lower().strip(".,!?") for w in resp_words if w.isalpha()]
    if words_lower:
        ttr = len(set(words_lower)) / len(words_lower)
    else:
        ttr = 0.5
    ppl_proxy = min(ttr * 1.5, 1.0)   # scale TTR → [0,1]

    score = (WEIGHT_COMPLEXITY * complexity +
             WEIGHT_AMBIGUITY  * ambiguity  +
             WEIGHT_PPL_PROXY  * ppl_proxy)

    return {
        "difficulty"  : round(score, 4),
        "complexity"  : round(complexity, 4),
        "ambiguity"   : round(ambiguity, 4),
        "ppl_proxy"   : round(ppl_proxy, 4),
    }


def assign_lfr_phase(examples: list) -> list:
    """
    Assign easy/medium/hard by percentile split (33rd / 67th).
    Guarantees no empty phase regardless of difficulty distribution.
    """
    import numpy as np
    scores = [ex["difficulty"] for ex in examples]
    p33    = np.percentile(scores, 33.33)
    p67    = np.percentile(scores, 66.67)

    for ex in examples:
        d = ex["difficulty"]
        if d <= p33:
            ex["lfr_phase"] = "easy"
        elif d <= p67:
            ex["lfr_phase"] = "medium"
        else:
            ex["lfr_phase"] = "hard"

    counts = Counter(ex["lfr_phase"] for ex in examples)
    print(f"  LFR phases — easy:{counts['easy']}  medium:{counts['medium']}  hard:{counts['hard']}")
    assert counts["easy"] > 0 and counts["medium"] > 0 and counts["hard"] > 0, \
        "Empty LFR phase — should never happen with percentile split!"
    return examples


print("Quality filter + difficulty scorer defined ✅")

Quality filter + difficulty scorer defined ✅


In [5]:
# CELL 5 — Build CONTROL dataset from Stanford Alpaca
# Alpaca 52K: GPT-3 generated, inconsistent quality — realistic low-quality baseline
from datasets import load_dataset
from tqdm import tqdm
import json, os

print("Loading Stanford Alpaca...")
alpaca_raw = load_dataset("tatsu-lab/alpaca", split="train")
print(f"  Raw Alpaca: {len(alpaca_raw):,} examples")

control_examples = []
for ex in tqdm(alpaca_raw, desc="Processing Alpaca"):
    instruction = ex.get("instruction", "").strip()
    inp         = ex.get("input", "").strip()
    response    = ex.get("output", "").strip()

    # Basic sanity checks (no quality filter — this is the control!)
    if not instruction or not response:
        continue
    if len(response.split()) < 10:   # skip trivially short responses
        continue
    if len(response.split()) > 1000: # skip unusually long outliers
        continue

    # Combine instruction + input context
    full_instruction = instruction
    if inp:
        full_instruction = f"{instruction}\n\nContext: {inp}"

    diff = difficulty_score(full_instruction, response)

    control_examples.append({
        "instruction" : full_instruction,
        "response"    : response,
        "source"      : "alpaca",
        "quality"     : "control",
        "lfr_phase"   : None,          # no LFR for control in E1 — added when used in E3
        **diff,
    })

# Shuffle and cap at target
random.shuffle(control_examples)
control_examples = control_examples[:CONTROL_TARGET]

# Assign LFR phases for E3 usage
control_examples = assign_lfr_phase(control_examples)

print(f"\nControl dataset: {len(control_examples):,} examples")
print(f"  Avg response words : {sum(len(e['response'].split()) for e in control_examples)/len(control_examples):.0f}")
print(f"  Avg difficulty     : {sum(e['difficulty'] for e in control_examples)/len(control_examples):.3f}")

# Write to file
control_path = os.path.join(WRITE_DIR, "control_sft.jsonl")
with open(control_path, "w") as f:
    for ex in control_examples:
        f.write(json.dumps(ex) + "\n")
print(f"\n✅ Written → {control_path}  ({os.path.getsize(control_path)/1e6:.1f} MB)")

Loading Stanford Alpaca...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

  Raw Alpaca: 52,002 examples


Processing Alpaca: 100%|██████████| 52002/52002 [00:04<00:00, 12576.31it/s]


  LFR phases — easy:6668  medium:6689  hard:6643

Control dataset: 20,000 examples
  Avg response words : 58
  Avg difficulty     : 0.551

✅ Written → /kaggle/working/control_sft.jsonl  (12.5 MB)


In [6]:
# CELL 6 — Build GOLDEN dataset: LIMA + OpenHermes-2.5 + Dolly-15K
# All three sources are quality-filtered and combined into one curated pool

golden_raw = []

# ── Source 1: LIMA (1,000 hand-curated examples) ─────────────────────────────
print("Loading LIMA...")
try:
    lima = load_dataset("GAIR/lima", split="train")
    lima_added = 0
    for ex in tqdm(lima, desc="LIMA"):
        convs = ex.get("conversations", [])
        if len(convs) < 2:
            continue
        instruction = convs[0].strip()
        response    = convs[1].strip()
        if not passes_quality_filter(response):
            continue
        diff = difficulty_score(instruction, response)
        golden_raw.append({
            "instruction" : instruction,
            "response"    : response,
            "source"      : "lima",
            "quality"     : "gold",
            **diff,
        })
        lima_added += 1
    print(f"  LIMA: {lima_added} examples kept (of {len(lima)})")
except Exception as e:
    print(f"  ⚠️  LIMA load failed: {e} — continuing without it")
    print("     (Accept the GAIR/lima license at huggingface.co/datasets/GAIR/lima)")

# ── Source 2: OpenHermes-2.5 (GPT-4 generated, filtered) ────────────────────
print("\nLoading OpenHermes-2.5...")
try:
    oh = load_dataset("teknium/OpenHermes-2.5", split="train", streaming=True)
    oh_pool  = []
    oh_added = 0
    for ex in tqdm(oh, desc="OpenHermes sampling", total=OH_SAMPLE):
        if len(oh_pool) >= OH_SAMPLE:
            break
        convs = ex.get("conversations", [])
        if len(convs) < 2:
            continue
        instruction = convs[0].get("value", "").strip()
        response    = convs[1].get("value", "").strip()
        if not instruction or not response:
            continue
        oh_pool.append({"instruction": instruction, "response": response})

    print(f"  OpenHermes sampled: {len(oh_pool):,}")

    random.shuffle(oh_pool)
    for item in tqdm(oh_pool, desc="OpenHermes filter"):
        if oh_added >= OH_KEEP_MAX:
            break
        if not passes_quality_filter(item["response"]):
            continue
        diff = difficulty_score(item["instruction"], item["response"])
        golden_raw.append({
            "instruction" : item["instruction"],
            "response"    : item["response"],
            "source"      : "openhermes",
            "quality"     : "gold",
            **diff,
        })
        oh_added += 1
    print(f"  OpenHermes: {oh_added} examples kept")
except Exception as e:
    print(f"  ⚠️  OpenHermes load failed: {e}")

# ── Source 3: Dolly-15K (human-written, filtered) ────────────────────────────
print("\nLoading Dolly-15K...")
try:
    dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
    dolly_added = 0
    for ex in tqdm(dolly, desc="Dolly"):
        if ex.get("category") not in DOLLY_CATEGORIES:
            continue
        instruction = ex.get("instruction", "").strip()
        context     = ex.get("context", "").strip()
        response    = ex.get("response", "").strip()

        if not instruction or not response:
            continue
        if len(response.split()) < DOLLY_MIN_WORDS:
            continue

        full_instruction = instruction
        if context:
            full_instruction = f"{instruction}\n\nContext: {context}"

        if not passes_quality_filter(response):
            continue

        diff = difficulty_score(full_instruction, response)
        golden_raw.append({
            "instruction" : full_instruction,
            "response"    : response,
            "source"      : "dolly",
            "quality"     : "gold",
            **diff,
        })
        dolly_added += 1
    print(f"  Dolly: {dolly_added} examples kept")
except Exception as e:
    print(f"  ⚠️  Dolly load failed: {e}")

print(f"\nGolden raw pool: {len(golden_raw):,} examples")
print("Source breakdown:")
for src in ["lima", "openhermes", "dolly"]:
    n = sum(1 for e in golden_raw if e["source"] == src)
    print(f"  {src:<15}: {n:,}")

Loading LIMA...


README.md:   0%|          | 0.00/368 [00:00<?, ?B/s]

  ⚠️  LIMA load failed: Dataset 'GAIR/lima' is a gated dataset on the Hub. You must be authenticated to access it. — continuing without it
     (Accept the GAIR/lima license at huggingface.co/datasets/GAIR/lima)

Loading OpenHermes-2.5...


README.md: 0.00B [00:00, ?B/s]

OpenHermes sampling: 100%|██████████| 25000/25000 [00:52<00:00, 471.73it/s] 


...

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Dolly: 100%|██████████| 15011/15011 [00:00<00:00, 20959.50it/s]

  Dolly: 280 examples kept

Golden raw pool: 4,873 examples
Source breakdown:
  lima           : 0
  openhermes     : 4,593
  dolly          : 280


In [7]:
# CELL 7 — Finalise golden dataset + assign LFR phases
import numpy as np

# Validation: check we have enough data
if len(golden_raw) < 1000:
    raise RuntimeError(
        f"Golden pool only has {len(golden_raw)} examples. "
        "At least 1000 required. Check HF login and dataset availability above."
    )

# Shuffle before capping
random.shuffle(golden_raw)
golden_examples = golden_raw[:GOLDEN_TARGET]

print(f"Golden dataset: {len(golden_examples):,} examples (capped from {len(golden_raw):,})")

# Assign LFR difficulty phases via percentile split
golden_examples = assign_lfr_phase(golden_examples)

# Stats
print(f"\nDifficulty distribution:")
scores = [e["difficulty"] for e in golden_examples]
print(f"  Min  : {min(scores):.3f}")
print(f"  Mean : {np.mean(scores):.3f}")
print(f"  Max  : {max(scores):.3f}")
print(f"  p33  : {np.percentile(scores, 33):.3f}")
print(f"  p67  : {np.percentile(scores, 67):.3f}")
print(f"\nAvg response words : {sum(len(e['response'].split()) for e in golden_examples)/len(golden_examples):.0f}")

# Source breakdown after cap
print("\nSource breakdown (final):")
for src in ["lima", "openhermes", "dolly"]:
    n = sum(1 for e in golden_examples if e["source"] == src)
    print(f"  {src:<15}: {n:,}  ({100*n/len(golden_examples):.1f}%)")

# Write to file
golden_path = os.path.join(WRITE_DIR, "golden_sft.jsonl")
with open(golden_path, "w") as f:
    for ex in golden_examples:
        f.write(json.dumps(ex) + "\n")

print(f"\n✅ Written → {golden_path}  ({os.path.getsize(golden_path)/1e6:.1f} MB)")

Golden dataset: 4,873 examples (capped from 4,873)
  LFR phases — easy:1625  medium:1625  hard:1623

Difficulty distribution:
  Min  : 0.221
  Mean : 0.446
  Max  : 0.734
  p33  : 0.383
  p67  : 0.491

Avg response words : 280

Source breakdown (final):
  lima           : 0  (0.0%)
  openhermes     : 4,593  (94.3%)
  dolly          : 280  (5.7%)

✅ Written → /kaggle/working/golden_sft.jsonl  (11.6 MB)


In [8]:
# CELL 8 — Build VALIDATION set from LIMA test split
# This is the neutral validation set used during ALL training experiments.
# It is never part of training — LIMA test is a clean held-out split.

print("Loading LIMA test split for validation...")

val_examples = []

try:
    lima_test = load_dataset("GAIR/lima", split="test")
    print(f"  LIMA test: {len(lima_test)} examples")

    for ex in lima_test:
        convs = ex.get("conversations", [])
        if len(convs) < 2:
            continue
        instruction = convs[0].strip()
        response    = convs[1].strip()
        if not instruction or not response:
            continue

        val_examples.append({
            "instruction" : instruction,
            "response"    : response,
            "source"      : "lima_test",
        })

    print(f"  Val examples from LIMA test: {len(val_examples)}")

except Exception as e:
    print(f"  ⚠️  LIMA test load failed: {e}")
    print("  Falling back to Dolly open_qa subset as validation...")

    # Fallback: use Dolly open_qa (not in golden training set since we filtered by different categories)
    try:
        dolly_val = load_dataset("databricks/databricks-dolly-15k", split="train")
        fallback  = [
            {"instruction": ex["instruction"],
             "response"   : ex["response"],
             "source"     : "dolly_val"}
            for ex in dolly_val
            if ex.get("category") == "open_qa"
            and len(ex.get("response","").split()) >= 30
        ]
        random.shuffle(fallback)
        val_examples = fallback[:VAL_SIZE]
        print(f"  Fallback val set: {len(val_examples)} examples")
    except Exception as e2:
        raise RuntimeError(f"Could not build val set: LIMA failed ({e}), Dolly fallback failed ({e2})")

# Cap to VAL_SIZE
random.shuffle(val_examples)
val_examples = val_examples[:VAL_SIZE]

print(f"\nValidation set: {len(val_examples)} examples")
print(f"  Avg response words: {sum(len(e['response'].split()) for e in val_examples)/len(val_examples):.0f}")

# Write to file
val_path = os.path.join(WRITE_DIR, "val_sft.jsonl")
with open(val_path, "w") as f:
    for ex in val_examples:
        f.write(json.dumps(ex) + "\n")

print(f"\n✅ Written → {val_path}  ({os.path.getsize(val_path)/1e6:.2f} MB)")

Loading LIMA test split for validation...
  ⚠️  LIMA test load failed: Dataset 'GAIR/lima' is a gated dataset on the Hub. You must be authenticated to access it.
  Falling back to Dolly open_qa subset as validation...
  Fallback val set: 300 examples

Validation set: 300 examples
  Avg response words: 98

✅ Written → /kaggle/working/val_sft.jsonl  (0.21 MB)


In [9]:
# CELL 9 — Verification: load all three files and confirm format
import json, os

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

control_check = load_jsonl(os.path.join(WRITE_DIR, "control_sft.jsonl"))
golden_check  = load_jsonl(os.path.join(WRITE_DIR, "golden_sft.jsonl"))
val_check     = load_jsonl(os.path.join(WRITE_DIR, "val_sft.jsonl"))

print("=" * 60)
print("DATASET VERIFICATION")
print("=" * 60)

for name, data in [("control_sft", control_check), ("golden_sft", golden_check), ("val_sft", val_check)]:
    print(f"\n{name}:")
    print(f"  Examples    : {len(data):,}")
    print(f"  Fields      : {list(data[0].keys())}")
    print(f"  Avg resp len: {sum(len(e['response'].split()) for e in data)/len(data):.0f} words")
    assert all("instruction" in e and "response" in e for e in data), f"{name}: missing required fields!"
    assert all(e["instruction"] and e["response"] for e in data), f"{name}: empty instruction/response!"
    print(f"  Format      : ✅")

# Check LFR phases in golden
for phase in ["easy", "medium", "hard"]:
    n = sum(1 for e in golden_check if e.get("lfr_phase") == phase)
    print(f"\ngolden LFR '{phase}': {n:,} examples")
    assert n > 0, f"Empty LFR phase '{phase}' — critical error!"

# Check LFR phases in control
for phase in ["easy", "medium", "hard"]:
    n = sum(1 for e in control_check if e.get("lfr_phase") == phase)
    print(f"control LFR '{phase}': {n:,} examples")
    assert n > 0, f"Empty LFR phase '{phase}' in control — critical error!"

print("\n" + "=" * 60)
print("ALL CHECKS PASSED ✅")
print("=" * 60)

DATASET VERIFICATION

control_sft:
  Examples    : 20,000
  Fields      : ['instruction', 'response', 'source', 'quality', 'lfr_phase', 'difficulty', 'complexity', 'ambiguity', 'ppl_proxy']
  Avg resp len: 58 words
  Format      : ✅

golden_sft:
  Examples    : 4,873
  Fields      : ['instruction', 'response', 'source', 'quality', 'difficulty', 'complexity', 'ambiguity', 'ppl_proxy', 'lfr_phase']
  Avg resp len: 280 words
  Format      : ✅

val_sft:
  Examples    : 300
  Fields      : ['instruction', 'response', 'source']
  Avg resp len: 98 words
  Format      : ✅

golden LFR 'easy': 1,625 examples

golden LFR 'medium': 1,625 examples

golden LFR 'hard': 1,623 examples
control LFR 'easy': 6,668 examples
control LFR 'medium': 6,689 examples
control LFR 'hard': 6,643 examples

ALL CHECKS PASSED ✅


In [10]:
# CELL 10 — Print sample examples for sanity check
import json

print("=" * 60)
print("SAMPLE: Control (Alpaca)")
print("=" * 60)
ex = control_check[0]
print(f"Instruction: {ex['instruction'][:200]}")
print(f"Response   : {ex['response'][:300]}...")
print(f"Difficulty : {ex['difficulty']}  Phase: {ex['lfr_phase']}")
print(f"Source     : {ex['source']}")

print("\n" + "=" * 60)
print("SAMPLE: Golden (by phase)")
print("=" * 60)
for phase in ["easy", "medium", "hard"]:
    ex = next(e for e in golden_check if e.get("lfr_phase") == phase)
    print(f"\n[{phase.upper()}] (difficulty={ex['difficulty']}, source={ex['source']})")
    print(f"Instruction: {ex['instruction'][:150]}")
    print(f"Response   : {ex['response'][:200]}...")

print("\n" + "=" * 60)
print("SAMPLE: Validation")
print("=" * 60)
ex = val_check[0]
print(f"Instruction: {ex['instruction'][:200]}")
print(f"Response   : {ex['response'][:300]}...")

SAMPLE: Control (Alpaca)
Instruction: Edit the given sentence for correct grammar usage.

Context: I had so much fun on our vacation what a wonderful time we had!
Response   : I had so much fun on our vacation! What a wonderful time we had!...
Difficulty : 0.4632  Phase: easy
Source     : alpaca

SAMPLE: Golden (by phase)

[EASY] (difficulty=0.3104, source=openhermes)
Instruction: In a town, there are 10 houses on one side of the street and 15 houses on the other side. Each house has a dog. One day, all dogs from the first side 
Response   : There are still 10 dogs in the first yard and 15 dogs in the second yard.

Here's the reasoning:

1. Initially, there are 10 dogs in the first yard (from the 10 houses on that side) and 15 dogs in the...

[MEDIUM] (difficulty=0.4189, source=openhermes)
Instruction: A pharmaceutical company is considering the acquisition of a smaller biotech firm that specializes in gene therapy research. The acquisition price is 
Response   : The payback period is 

In [11]:
# CELL 11 — Final summary + instructions
import os

files = ["control_sft.jsonl", "golden_sft.jsonl", "val_sft.jsonl"]

print("=" * 60)
print("DATASET BUILD COMPLETE")
print("=" * 60)
for fname in files:
    fpath = os.path.join(WRITE_DIR, fname)
    size  = os.path.getsize(fpath) / 1e6
    lines = sum(1 for _ in open(fpath))
    print(f"  {fname:<25} {lines:>7,} examples  {size:>6.1f} MB")

print("\n" + "=" * 60)
print("NEXT STEPS")
print("=" * 60)
print("""
1. Click [Save Version] → [Save & Run All (Commit)]
   Wait for the commit to finish (~5 min)

2. Go to your notebook's Output tab
   Click the three dots next to the output folder
   Click [New Dataset] → name it exactly: sft-datasets
   Wait for upload (~5-10 min depending on file size)

3. In sft_combined_all6.ipynb:
   Click [Add Data] → search for 'sft-datasets'
   Add it — it will mount at /kaggle/input/sft-datasets/

4. Run sft_combined_all6.ipynb
   Cell 4 will load all three files from /kaggle/input/sft-datasets/
   If Cell 4 fails with FileNotFoundError, the mount path is wrong.
   Run: import os; print(os.listdir('/kaggle/input/sft-datasets/'))
   to confirm the files are there.
""")
print("✅ NB0 complete — save output as Kaggle Dataset: 'sft-datasets'")

DATASET BUILD COMPLETE
  control_sft.jsonl          20,000 examples    12.5 MB
  golden_sft.jsonl            4,873 examples    11.6 MB
  val_sft.jsonl                 300 examples     0.2 MB

NEXT STEPS

1. Click [Save Version] → [Save & Run All (Commit)]
   Wait for the commit to finish (~5 min)

2. Go to your notebook's Output tab
   Click the three dots next to the output folder
   Click [New Dataset] → name it exactly: sft-datasets
   Wait for upload (~5-10 min depending on file size)

3. In sft_combined_all6.ipynb:
   Click [Add Data] → search for 'sft-datasets'
   Add it — it will mount at /kaggle/input/sft-datasets/

4. Run sft_combined_all6.ipynb
   Cell 4 will load all three files from /kaggle/input/sft-datasets/
   If Cell 4 fails with FileNotFoundError, the mount path is wrong.
   Run: import os; print(os.listdir('/kaggle/input/sft-datasets/'))
   to confirm the files are there.

✅ NB0 complete — save output as Kaggle Dataset: 'sft-datasets'
